In [ ]:
!pip install weaviate-client requests --quiet

# Environment
In order for this to run, both the client and server must have access to Keycloack from a single url.
For a Mac with Rancher Desktop, for example, you can add this to your /etc/hosts

```
127.0.0.1 	host.docker.internal
```

First, run only keycloack, otherwise Weaviate will not be up due to no helm created

In [ ]:
!docker compose up -d keycloack

Run this idempotent code. It will:
Configures:
  1. Realm 'weaviate'
  2. Public client 'weaviate' with Direct Access Grants + Audience mapper
     + Group Membership mapper (groups appear in JWT as 'groups' claim)
  3. User profile: removes firstName/lastName as required (Keycloak 26 quirk)
  4. User 'admin-user' (password: test)  — Weaviate RBAC root user
  5. Group 'SpecialCollections'
  6. User 'newuser' (password: newuser)  — member of SpecialCollections
  7. Create group 'SpecialCollections'
  8. Create newuser and add to SpecialCollections

In [1]:
import requests

KEYCLOAK   = "http://host.docker.internal:8081"
REALM      = "weaviate"
CLIENT_ID  = "weaviate"
ADMIN_USER = "admin-user"
ADMIN_PASS = "test"
NEW_USER   = "newuser"
NEW_PASS   = "newuser"
GROUP_NAME = "SpecialCollections"

def admin_token():
    r = requests.post(
        f"{KEYCLOAK}/realms/master/protocol/openid-connect/token",
        data={"grant_type": "password", "client_id": "admin-cli",
              "username": "admin", "password": "admin"},
    )
    r.raise_for_status()
    return r.json()["access_token"]

def adm(method, path, **kw):
    return requests.request(
        method, f"{KEYCLOAK}/admin/{path}",
        headers={"Authorization": f"Bearer {admin_token()}",
                 "Content-Type": "application/json"},
        **kw,
    )

# 1. Create realm
r = adm("POST", "realms", json={"realm": REALM, "enabled": True})
print(f"[1] Create realm:            {r.status_code}  (409 = already exists, ok)")

# 2. Remove firstName / lastName / email as required in User Profile.
#    Keycloak 26 marks them required for the 'user' role by default —
#    this causes "Account is not fully set up" on Resource Owner Password login.
profile = adm("GET", f"realms/{REALM}/users/profile").json()
for attr in profile.get("attributes", []):
    if attr.get("name") in ("firstName", "lastName", "email"):
        attr.pop("required", None)
r = adm("PUT", f"realms/{REALM}/users/profile", json=profile)
print(f"[2] Patch user profile:      {r.status_code}")

# 3. Create public client with Direct Access Grants enabled
r = adm("POST", f"realms/{REALM}/clients", json={
    "clientId": CLIENT_ID,
    "enabled": True,
    "publicClient": True,
    "directAccessGrantsEnabled": True,
    "protocol": "openid-connect",
})
print(f"[3] Create client:           {r.status_code}  (409 = already exists, ok)")

client_uuid = adm("GET", f"realms/{REALM}/clients?clientId={CLIENT_ID}").json()[0]["id"]

# 4. Add Audience mapper
r = adm("POST", f"realms/{REALM}/clients/{client_uuid}/protocol-mappers/models", json={
    "name": "weaviate-audience",
    "protocol": "openid-connect",
    "protocolMapper": "oidc-audience-mapper",
    "config": {
        "included.client.audience": CLIENT_ID,
        "id.token.claim": "false",
        "access.token.claim": "true",
    },
})
print(f"[4] Add audience mapper:     {r.status_code}  (409 = already exists, ok)")

# 5. Add Group Membership mapper (full.path=true → '/SpecialCollections' in JWT)
r = adm("POST", f"realms/{REALM}/clients/{client_uuid}/protocol-mappers/models", json={
    "name": "weaviate-groups",
    "protocol": "openid-connect",
    "protocolMapper": "oidc-group-membership-mapper",
    "config": {
        "claim.name": "groups",
        "full.path": "true",
        "id.token.claim": "false",
        "access.token.claim": "true",
        "userinfo.token.claim": "false",
    },
})
print(f"[5] Add groups mapper:       {r.status_code}  (409 = already exists, ok)")

# 6. Create admin user
r = adm("POST", f"realms/{REALM}/users", json={
    "username": ADMIN_USER,
    "enabled": True,
    "emailVerified": True,
    "credentials": [{"type": "password", "value": ADMIN_PASS, "temporary": False}],
})
print(f"[6] Create admin-user:       {r.status_code}  (409 = already exists, ok)")

# 7. Create group 'SpecialCollections'
r = adm("POST", f"realms/{REALM}/groups", json={"name": GROUP_NAME})
print(f"[7] Create group:            {r.status_code}  (409 = already exists, ok)")
group_id = adm("GET", f"realms/{REALM}/groups?search={GROUP_NAME}").json()[0]["id"]
print(f"    Group UUID: {group_id}")

# 8. Create newuser and add to SpecialCollections
r = adm("POST", f"realms/{REALM}/users", json={
    "username": NEW_USER,
    "enabled": True,
    "emailVerified": True,
    "credentials": [{"type": "password", "value": NEW_PASS, "temporary": False}],
})
print(f"[8] Create newuser:          {r.status_code}  (409 = already exists, ok)")
new_user_id = adm("GET", f"realms/{REALM}/users?username={NEW_USER}").json()[0]["id"]

r = adm("PUT", f"realms/{REALM}/users/{new_user_id}/groups/{group_id}")
print(f"    Add newuser to group:    {r.status_code}")

print("\nKeycloak ready.")

[1] Create realm:            201  (409 = already exists, ok)
[2] Patch user profile:      200
[3] Create client:           201  (409 = already exists, ok)
[4] Add audience mapper:     201  (409 = already exists, ok)
[5] Add groups mapper:       201  (409 = already exists, ok)
[6] Create admin-user:       201  (409 = already exists, ok)
[7] Create group:            201  (409 = already exists, ok)
    Group UUID: 05820353-d50f-4c4d-82c2-7451f3ae99cb
[8] Create newuser:          201  (409 = already exists, ok)
    Add newuser to group:    204

Keycloak ready.


# Now, Run Weaviate
We will connect as the admin user set roles and group setup, then create some collections.

In [ ]:
!docker compose up -d weaviate

Now let's connect with admin user, in order to create the RBAC roles and groups

In [ ]:
import weaviate
from weaviate.classes.init import Auth

# Weaviate is on localhost:8080, Keycloak is on localhost:8081
# Auth.client_password uses the OIDC Resource Owner Password flow.
# The token is fetched from Keycloak using the credentials below,
# then passed as a Bearer token to Weaviate.
# this means both client and server must have acess to OIDC server (Keycloak)
# for token fetching and verification.
client = weaviate.connect_to_local(
    auth_credentials=Auth.client_password(
        username="admin-user",
        password="test"
    )
)

print(client.is_ready())

In [ ]:
import weaviate
from weaviate.classes.init import Auth
from weaviate.classes.rbac import Permissions

# Connect as admin-user (RBAC root) to manage roles
admin_client = weaviate.connect_to_local(
    auth_credentials=Auth.client_password(
        username="admin-user", password="test"
    ),
)

ROLE_NAME  = "special-collections-role"
GROUP_NAME = "/SpecialCollections"  # Keycloak full path — matches JWT 'groups' claim

# Create a role with read access to all collections
try:
    admin_client.roles.create(
        role_name=ROLE_NAME,
        permissions=[
            Permissions.collections(collection="Special*", read_config=True),
            Permissions.data(collection="Special*", read=True),
        ],
    )
    print(f"Role '{ROLE_NAME}' created.")
except Exception as e:
    print(f"Role create: {e}")

# Assign the role to the group
try:
    admin_client.groups.oidc.assign_roles(group_id=GROUP_NAME, role_names=[ROLE_NAME])
    print(f"Role '{ROLE_NAME}' assigned to group '{GROUP_NAME}'.")
except Exception as e:
    print(f"Assign error: {e}")

# Verify
try:
    role = admin_client.roles.get(ROLE_NAME)
    print("\nRole details:", role)
except Exception as e:
    print(f"Verify error: {e}")

# Let's create some collections
for col in ["SpecialCollection1", "SpecialCollection2", "NotSpecialCollection1"]:
    try:
        admin_client.collections.create(col)
        print(f"Collection '{col}' created.")
    except Exception as e:
        print(f"Collection create error for '{col}': {e}")

admin_client.close()


In [ ]:
# now connect with our user
client = weaviate.connect_to_local(
    auth_credentials=Auth.client_password(
        username="newuser",
        password="newuser"
    )
)
collection_keys = client.collections.list_all().keys()
print("\nCollections visible to newuser:", collection_keys)
assert "NotSpecialCollection1" not in collection_keys, "newuser should NOT see NotSpecialCollection1"
assert "SpecialCollection1" in collection_keys, "newuser should see SpecialCollection1"
assert "SpecialCollection2" in collection_keys, "newuser should see SpecialCollection2"

# TLS / Self-Signed Certificate Test

This section tests Weaviate connectivity to a **HTTPS Keycloak** that uses a **self-signed certificate**.

Two modes are supported — both are demonstrated here:

| Mode | Weaviate env var | Image requirement |
|------|-----------------|-------------------|
| **A** | `AUTHENTICATION_OIDC_CERTIFICATE=<PEM>` | Any stable release |
| **B** | `AUTHENTICATION_OIDC_INSECURE_SKIP_TLS_VERIFY=true` | Preview build from [PR #10813](https://github.com/weaviate/weaviate/pull/10813) |

The `docker-compose-tls.yml` defaults to **Mode B** with the preview image:
```
semitechnologies/weaviate:preview-implements-insecure-skip-oidc-tls-verify-d0626f9.arm64
```

Key things being validated end-to-end:
- Keycloak is configured with a self-signed TLS cert for `host.docker.internal`
- Weaviate skips TLS verification for OIDC via `AUTHENTICATION_OIDC_INSECURE_SKIP_TLS_VERIFY=true`
- OIDC token flow and RBAC still work correctly

**Prerequisites:** `openssl` must be available on the host (`which openssl`).

In [24]:
# Step 1: Generate self-signed CA + Keycloak server certificate.
# Safe to re-run — overwrites existing certs in ./certs/
import os, subprocess

os.makedirs("certs", exist_ok=True)
result = subprocess.run(["bash", "certs/generate.sh"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Certificate generation failed")

==> Generating CA key and self-signed certificate...
    ca.pem created
==> Generating server key...
    server-key.pem created
==> Writing OpenSSL SAN config...
==> Generating server CSR...
==> Signing server certificate with CA...
    server.pem created

Done. Certificates generated in: /Users/dudanogueira/dev/weaviate/weaviate-keycloack/certs

  ca.pem         — CA cert  → AUTHENTICATION_OIDC_CERTIFICATE
  server.pem     — Keycloak server cert (signed by CA)
  server-key.pem — Keycloak server private key

Verify SAN:
            X509v3 Subject Alternative Name: 
                DNS:host.docker.internal, DNS:localhost, IP Address:127.0.0.1



In [ ]:
# Step 2: Tear down any previous TLS stack, then start the TLS stack.
# The CA PEM is injected into the shell environment so docker-compose can
# substitute ${AUTHENTICATION_OIDC_CERTIFICATE} in docker-compose-tls.yml.
import os, subprocess

ca_pem = open("certs/ca.pem").read()
env = {**os.environ, "AUTHENTICATION_OIDC_CERTIFICATE": ca_pem}

# Tear down cleanly first (idempotent)
subprocess.run(
    ["docker", "compose", "-f", "docker-compose-tls.yml", "--project-name", "weaviate-tls", "down", "-v"],
    env=env, capture_output=True,
)

result = subprocess.run(
    ["docker", "compose", "-f", "docker-compose-tls.yml", "--project-name", "weaviate-tls", "up", "-d", "keycloack"],
    env=env, capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("docker compose up failed")

/var/folders/4j/js2lp8b17zs2cvk9v9pl26cc0000gn/T/ipykernel_65772/2887288845.py:6: ResourceWarning: unclosed file <_io.TextIOWrapper name='certs/ca.pem' mode='r' encoding='UTF-8'>
  ca_pem = open("certs/ca.pem").read()


In [34]:
# Step 3: Patch SSL + wait for Keycloak HTTPS.
#
# The weaviate Python client uses httpx, which calls ssl.create_default_context()
# for its OIDC discovery requests to Keycloak. We patch that function to load
# our custom CA so every HTTPS connection in this process trusts it.
import os, ssl, time, requests

CA_CERT  = os.path.abspath("certs/ca.pem")
KC_HTTPS = "https://host.docker.internal:8443"

# Patch ssl.create_default_context to always load our custom CA.
# This affects httpx, requests, and any other lib using the standard ssl module.
_orig_create_default_context = ssl.create_default_context
def _patched_create_default_context(*args, **kwargs):
    ctx = _orig_create_default_context(*args, **kwargs)
    ctx.load_verify_locations(cafile=CA_CERT)
    return ctx
ssl.create_default_context = _patched_create_default_context

print("SSL context patched to trust custom CA.")

print("Waiting for Keycloak HTTPS...")
for i in range(40):
    try:
        r = requests.get(f"{KC_HTTPS}/realms/master", verify=CA_CERT, timeout=3)
        if r.status_code < 500:
            print(f"  Keycloak HTTPS ready (HTTP {r.status_code})")
            break
    except Exception as e:
        print(f"  attempt {i+1}... {type(e).__name__}: {e}")
    time.sleep(3)
else:
    raise TimeoutError("Keycloak HTTPS did not become ready")

SSL context patched to trust custom CA.
Waiting for Keycloak HTTPS...
  Keycloak HTTPS ready (HTTP 200)


In [35]:
# Step 4: Configure Keycloak (idempotent).
# Admin API calls use the HTTP port (8081 → 8080 inside Keycloak).
# This avoids CA cert complexity for admin setup while keeping OIDC on HTTPS.
import requests

KEYCLOAK   = "http://host.docker.internal:8081"
REALM      = "weaviate"
CLIENT_ID  = "weaviate"
ADMIN_USER = "admin-user"
ADMIN_PASS = "test"
NEW_USER   = "newuser"
NEW_PASS   = "newuser"
GROUP_NAME = "SpecialCollections"

def admin_token():
    r = requests.post(
        f"{KEYCLOAK}/realms/master/protocol/openid-connect/token",
        data={"grant_type": "password", "client_id": "admin-cli",
              "username": "admin", "password": "admin"},
    )
    r.raise_for_status()
    return r.json()["access_token"]

def adm(method, path, **kw):
    return requests.request(
        method, f"{KEYCLOAK}/admin/{path}",
        headers={"Authorization": f"Bearer {admin_token()}",
                 "Content-Type": "application/json"},
        **kw,
    )

r = adm("POST", "realms", json={"realm": REALM, "enabled": True})
print(f"[1] Create realm:            {r.status_code}  (409 = already exists, ok)")

profile = adm("GET", f"realms/{REALM}/users/profile").json()
for attr in profile.get("attributes", []):
    if attr.get("name") in ("firstName", "lastName", "email"):
        attr.pop("required", None)
r = adm("PUT", f"realms/{REALM}/users/profile", json=profile)
print(f"[2] Patch user profile:      {r.status_code}")

r = adm("POST", f"realms/{REALM}/clients", json={
    "clientId": CLIENT_ID, "enabled": True,
    "publicClient": True, "directAccessGrantsEnabled": True,
    "protocol": "openid-connect",
})
print(f"[3] Create client:           {r.status_code}  (409 = already exists, ok)")

client_uuid = adm("GET", f"realms/{REALM}/clients?clientId={CLIENT_ID}").json()[0]["id"]

r = adm("POST", f"realms/{REALM}/clients/{client_uuid}/protocol-mappers/models", json={
    "name": "weaviate-audience", "protocol": "openid-connect",
    "protocolMapper": "oidc-audience-mapper",
    "config": {"included.client.audience": CLIENT_ID,
               "id.token.claim": "false", "access.token.claim": "true"},
})
print(f"[4] Add audience mapper:     {r.status_code}  (409 = already exists, ok)")

r = adm("POST", f"realms/{REALM}/clients/{client_uuid}/protocol-mappers/models", json={
    "name": "weaviate-groups", "protocol": "openid-connect",
    "protocolMapper": "oidc-group-membership-mapper",
    "config": {"claim.name": "groups", "full.path": "true",
               "id.token.claim": "false", "access.token.claim": "true",
               "userinfo.token.claim": "false"},
})
print(f"[5] Add groups mapper:       {r.status_code}  (409 = already exists, ok)")

r = adm("POST", f"realms/{REALM}/users", json={
    "username": ADMIN_USER, "enabled": True, "emailVerified": True,
    "credentials": [{"type": "password", "value": ADMIN_PASS, "temporary": False}],
})
print(f"[6] Create admin-user:       {r.status_code}  (409 = already exists, ok)")

r = adm("POST", f"realms/{REALM}/groups", json={"name": GROUP_NAME})
print(f"[7] Create group:            {r.status_code}  (409 = already exists, ok)")
group_id = adm("GET", f"realms/{REALM}/groups?search={GROUP_NAME}").json()[0]["id"]

r = adm("POST", f"realms/{REALM}/users", json={
    "username": NEW_USER, "enabled": True, "emailVerified": True,
    "credentials": [{"type": "password", "value": NEW_PASS, "temporary": False}],
})
print(f"[8] Create newuser:          {r.status_code}  (409 = already exists, ok)")
new_user_id = adm("GET", f"realms/{REALM}/users?username={NEW_USER}").json()[0]["id"]
r = adm("PUT", f"realms/{REALM}/users/{new_user_id}/groups/{group_id}")
print(f"    Add newuser to group:    {r.status_code}")

print("\nKeycloak ready.")

[1] Create realm:            409  (409 = already exists, ok)
[2] Patch user profile:      200
[3] Create client:           409  (409 = already exists, ok)
[4] Add audience mapper:     409  (409 = already exists, ok)
[5] Add groups mapper:       409  (409 = already exists, ok)
[6] Create admin-user:       409  (409 = already exists, ok)
[7] Create group:            409  (409 = already exists, ok)
[8] Create newuser:          409  (409 = already exists, ok)
    Add newuser to group:    204

Keycloak ready.


In [2]:
# Step 5: Set up RBAC roles and collections via the API key (no OIDC needed for this).
import weaviate
from weaviate.classes.init import Auth
from weaviate.classes.rbac import Permissions

admin_client = weaviate.connect_to_local(
    auth_credentials=Auth.api_key("root-user-key"),
)

ROLE_NAME      = "special-collections-role"
KC_GROUP_CLAIM = "/SpecialCollections"  # full.path=true → leading slash

try:
    admin_client.roles.create(
        role_name=ROLE_NAME,
        permissions=[
            Permissions.collections(collection="Special*", read_config=True),
            Permissions.data(collection="Special*", read=True),
        ],
    )
    print(f"Role '{ROLE_NAME}' created.")
except Exception as e:
    print(f"Role create: {e}")

try:
    admin_client.groups.oidc.assign_roles(group_id=KC_GROUP_CLAIM, role_names=[ROLE_NAME])
    print(f"Role assigned to group '{KC_GROUP_CLAIM}'.")
except Exception as e:
    print(f"Assign error: {e}")

for col in ["SpecialCollection1", "SpecialCollection2", "NotSpecialCollection1"]:
    try:
        admin_client.collections.create(col)
        print(f"Collection '{col}' created.")
    except Exception as e:
        print(f"Collection '{col}': {e}")

admin_client.close()

Role 'special-collections-role' created.
Role assigned to group '/SpecialCollections'.
Collection 'SpecialCollection1' created.
Collection 'SpecialCollection2' created.
Collection 'NotSpecialCollection1' created.


In [37]:
# Step 6: Negative test — confirm the self-signed cert is NOT trusted by default.
# This is why AUTHENTICATION_OIDC_CERTIFICATE exists: without it Weaviate (and Python)
# would reject Keycloak's TLS handshake.
import requests

KC_HTTPS = "https://host.docker.internal:8443"

try:
    requests.get(f"{KC_HTTPS}/realms/weaviate", verify=True, timeout=5)
    print("UNEXPECTED: request succeeded without CA cert (should have failed!)")
except requests.exceptions.SSLError as e:
    print("Expected SSLError — self-signed cert is not in the default trust store.")
    print(f"  {type(e).__name__}: {str(e)[:120]}")
    print()
    print("=> This is the exact error Weaviate would get without AUTHENTICATION_OIDC_CERTIFICATE.")

# Now confirm it succeeds with our CA cert
r = requests.get(f"{KC_HTTPS}/realms/weaviate", verify="certs/ca.pem", timeout=5)
print(f"\nWith CA cert: HTTP {r.status_code} — TLS verified successfully.")

Expected SSLError — self-signed cert is not in the default trust store.
  SSLError: HTTPSConnectionPool(host='host.docker.internal', port=8443): Max retries exceeded with url: /realms/weaviate (Caused by 

=> This is the exact error Weaviate would get without AUTHENTICATION_OIDC_CERTIFICATE.

With CA cert: HTTP 200 — TLS verified successfully.


In [4]:
# MONKEYPATCH httpx

import httpx

# Monkey patch httpx to disable SSL verification globally
_original_init = httpx.Client.__init__
def _patched_init(self, *args, **kwargs):
    kwargs.setdefault("verify", False)
    _original_init(self, *args, **kwargs)
httpx.Client.__init__ = _patched_init

_original_async_init = httpx.AsyncClient.__init__
def _patched_async_init(self, *args, **kwargs):
    kwargs.setdefault("verify", False)
    _original_async_init(self, *args, **kwargs)
httpx.AsyncClient.__init__ = _patched_async_init

In [7]:
# Step 7: Fetch a token from Keycloak over HTTPS (using our CA cert),
# then pass it to Weaviate as a Bearer token.
# Weaviate validates the token by calling Keycloak's JWKS endpoint over HTTPS
# using the CA cert set in AUTHENTICATION_OIDC_CERTIFICATE — that's what we're testing.
import requests, base64, json, weaviate
from weaviate.classes.init import Auth

CA_CERT  = "certs/ca.pem"
KC_HTTPS = "https://host.docker.internal:8443"
REALM    = "weaviate"

def get_oidc_token(username, password):
    r = requests.post(
        f"{KC_HTTPS}/realms/{REALM}/protocol/openid-connect/token",
        data={"grant_type": "password", "client_id": "weaviate",
              "username": username, "password": password,
              "scope": "openid offline_access"},
        verify=CA_CERT,
    )
    r.raise_for_status()
    data = r.json()
    return data["access_token"], data.get("refresh_token")

def decode_claims(token):
    payload = token.split(".")[1]
    payload += "=" * (4 - len(payload) % 4)
    return json.loads(base64.b64decode(payload))

# -- admin-user --
admin_token, admin_refresh = get_oidc_token("admin-user", "test")
claims = decode_claims(admin_token)
print("admin-user token claims:")
print(f"  iss:                {claims.get('iss')}")
print(f"  preferred_username: {claims.get('preferred_username')}")
print(f"  aud:                {claims.get('aud')}")
print(f"  groups:             {claims.get('groups')}")

# Connect to Weaviate with the HTTPS-issued token.
# ssl.create_default_context is patched in Step 3 so the weaviate client's
# internal httpx calls to Keycloak (OIDC discovery/JWKS) are trusted.
client = weaviate.connect_to_local(
    auth_credentials=Auth.bearer_token(
        access_token=admin_token,
        refresh_token=admin_refresh,
    ),
)
print(f"\nWeaviate ready: {client.is_ready()}")
collections = list(client.collections.list_all().keys())
print(f"All collections (admin view): {collections}")
client.close()

admin-user token claims:
  iss:                https://host.docker.internal:8443/realms/weaviate
  preferred_username: admin-user
  aud:                ['weaviate', 'account']
  groups:             None

Weaviate ready: True
All collections (admin view): ['NotSpecialCollection1', 'SpecialCollection1', 'SpecialCollection2']


In [8]:
# Step 8: Verify RBAC still works correctly via HTTPS tokens.
# newuser is in /SpecialCollections and should only see Special* collections.
import weaviate
from weaviate.classes.init import Auth

newuser_token, newuser_refresh = get_oidc_token("newuser", "newuser")
newuser_claims = decode_claims(newuser_token)
print("newuser groups claim:", newuser_claims.get("groups"))

client = weaviate.connect_to_local(
    auth_credentials=Auth.bearer_token(
        access_token=newuser_token,
        refresh_token=newuser_refresh,
    ),
)
collection_keys = set(client.collections.list_all().keys())
print(f"\nCollections visible to newuser: {collection_keys}")

assert "SpecialCollection1"    in collection_keys, "newuser should see SpecialCollection1"
assert "SpecialCollection2"    in collection_keys, "newuser should see SpecialCollection2"
assert "NotSpecialCollection1" not in collection_keys, "newuser should NOT see NotSpecialCollection1"

print("\nAll RBAC assertions passed — TLS + self-signed cert + RBAC working correctly.")
client.close()

newuser groups claim: ['/SpecialCollections']

Collections visible to newuser: {'SpecialCollection1', 'SpecialCollection2'}

All RBAC assertions passed — TLS + self-signed cert + RBAC working correctly.


## Mode B: `AUTHENTICATION_OIDC_INSECURE_SKIP_TLS_VERIFY=true` (Preview Image)

# WARNING
## this should only be used for dev,tests, etc. 
# DO NOT USE IT IN PRODUCTION!

This mode tells Weaviate to **skip TLS certificate verification** when connecting to the OIDC provider (Keycloak). It is the simplest way to handle self-signed certs — no CA PEM needs to be injected into the container.

**Important:** this env var is only available in the preview build tied to [PR #10813](https://github.com/weaviate/weaviate/pull/10813):
```
semitechnologies/weaviate:preview-implements-insecure-skip-oidc-tls-verify-d0626f9.arm64
```

Once the PR is merged it will ship in a stable release.

### How it works

```
┌──────────────────────────────────────────────────────────────────┐
│  Weaviate container                                              │
│                                                                  │
│  AUTHENTICATION_OIDC_INSECURE_SKIP_TLS_VERIFY=true               │
│                                                                  │
│  OIDC discovery ──► https://host.docker.internal:8443/...        │
│                     (TLS cert NOT verified — any cert accepted)  │
└──────────────────────────────────────────────────────────────────┘
```

No CA cert is needed on the Weaviate side. The Python client side still needs the CA patched (Step 3) because `httpx` / `ssl` in the notebook process are separate from Weaviate's Go TLS stack.

### Comparison with Mode A (`AUTHENTICATION_OIDC_CERTIFICATE`)

| | Mode A | Mode B |
|---|---|---|
| Weaviate verifies Keycloak cert | Yes (via provided CA) | No (skipped) |
| Requires preview image | No | Yes (until PR merges) |
| Security | Better (cert pinned) | Lower (MITM possible) |
| Operational overhead | Must inject CA PEM at startup | Single env var |

Use **Mode A** in production. Use **Mode B** for dev/test environments where managing certs is impractical.

In [ ]:
# Step 9: Confirm Mode B is active — verify the running container uses the preview image
# and has AUTHENTICATION_OIDC_INSECURE_SKIP_TLS_VERIFY=true set.
import subprocess, json

result = subprocess.run(
    ["docker", "inspect", "--format", "{{json .Config}}", "weaviate-tls-weaviate-1"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print("Container not found — is the TLS stack running?")
    print(result.stderr)
else:
    cfg = json.loads(result.stdout)
    image = cfg.get("Image", "")
    env   = {e.split("=", 1)[0]: e.split("=", 1)[1] for e in cfg.get("Env", []) if "=" in e}

    print(f"Image:                        {image}")
    print(f"AUTHENTICATION_OIDC_INSECURE_SKIP_TLS_VERIFY: {env.get('AUTHENTICATION_OIDC_INSECURE_SKIP_TLS_VERIFY', '(not set)')}")
    print(f"AUTHENTICATION_OIDC_ISSUER:    {env.get('AUTHENTICATION_OIDC_ISSUER', '(not set)')}")
    print(f"AUTHENTICATION_OIDC_CERTIFICATE: {'(set — Mode A active)' if env.get('AUTHENTICATION_OIDC_CERTIFICATE') else '(not set)'}")

    assert env.get("AUTHENTICATION_OIDC_INSECURE_SKIP_TLS_VERIFY") == "true", \
        "Expected AUTHENTICATION_OIDC_INSECURE_SKIP_TLS_VERIFY=true"
    print("\nMode B confirmed: preview image + AUTHENTICATION_OIDC_INSECURE_SKIP_TLS_VERIFY=true")

Image:                        cr.weaviate.io/semitechnologies/weaviate:1.36.2
AUTHENTICATION_OIDC_INSECURE_SKIP_TLS_VERIFY: (not set)
AUTHENTICATION_OIDC_ISSUER:    https://host.docker.internal:8443/realms/weaviate
AUTHENTICATION_OIDC_CERTIFICATE: (set — Mode A active)


AssertionError: Expected preview image for AUTHENTICATION_OIDC_INSECURE_SKIP_TLS_VERIFY support

In [ ]:
# Step 10 (optional): Tear down the TLS stack and remove volumes.
import subprocess
result = subprocess.run(
    ["docker", "compose", "-f", "docker-compose-tls.yml", "--project-name", "weaviate-tls", "down", "-v"],
    capture_output=True, text=True,
)
print(result.stdout or result.stderr)